# LLM 6 — Is it good? Evaluating AI outputs

Changing a prompt and eyeballing one answer is guessing. Professionals write **evals**: tests for AI behavior. Today you write your first.

> **Working with your AI pair:** paste any error into your AI and ask *"explain this error like I'm new, then help me fix it — don't just give me the answer."*


In [ ]:
%pip install -q anthropic
import os, json, anthropic
from getpass import getpass
os.environ.setdefault('ANTHROPIC_API_KEY', getpass('Class API key: '))
MODEL='claude-opus-5'
client=anthropic.Anthropic()
def ask(prompt, system=None, max_tokens=300):
    kwargs=dict(model=MODEL,max_tokens=max_tokens,messages=[{'role':'user','content':prompt}])
    if system: kwargs['system']=system
    r=client.messages.create(**kwargs)
    return ''.join(b.text for b in r.content if b.type=='text')


## 1. The thing we're testing
A flashcard generator. Two candidate system prompts — which is better? Don't argue. **Measure.**


In [ ]:
prompt_A = 'Write one quiz question with answer about the given topic.'
prompt_B = ('Write one quiz question about the given topic for a high-school student. '
            'Format exactly as: Q: <question> then a new line A: <answer>. '
            'The question must be answerable from general knowledge, specific, and under 25 words.')


## 2. Checks — properties a good output must have
Each check is a plain function: output in, True/False out.


In [ ]:
def has_q_and_a(out): return 'Q:' in out and 'A:' in out
def question_short(out):
    q = out.split('A:')[0]
    return len(q.split()) <= 30
def not_empty(out): return len(out.strip()) > 10
CHECKS = [has_q_and_a, question_short, not_empty]


## 3. Run the eval — same topics through both prompts


In [ ]:
topics = ['the Great Migration','tokens in language models','photosynthesis','the 1919 Black Sox','fractions']
for name, sysprompt in [('A',prompt_A),('B',prompt_B)]:
    score = 0; total = 0
    for t in topics:
        out = ask(t, system=sysprompt)
        for c in CHECKS:
            total += 1; score += bool(c(out))
    print(f'prompt {name}: {score}/{total} checks passed')


## 4. What you just did
That's a real eval: fixed inputs, automatic checks, a number instead of a vibe. Every serious AI product runs thousands of these before changing a prompt. Also honest: checks test FORM well and TRUTH poorly — a wrong answer in perfect format passes. Truth-checking needs sources (course 1, lesson 6) or a human.


## 5. Your turn
Add two checks of your own (no hedging words? answer under 15 words?) and rerun. Did the winner change?


In [ ]:
# your checks


## 6. The build
Eval YOUR lesson-2 or lesson-5 tool: 5 fixed inputs, 3+ checks, scores before/after one prompt improvement. **Turn-in:** the score table and which prompt shipped.
